# Wine Quality: Exploratory Data Analysis

Phase 1 binary classification uses the UCI red-wine quality data. The target is `good = 1` when `quality >= 6`; `quality` itself is excluded from model features to prevent target leakage.

## Load data and inspect structure

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data" / "raw" / "winequality-red.csv").exists()
)
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "winequality-red.csv"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, sep=";")
df["good"] = (df["quality"] >= 6).astype(int)

print(f"Dataset path: {DATA_PATH}")
print(f"Shape (rows, columns): {df.shape}")
print("Data types:")
display(df.dtypes.rename("dtype").to_frame())
print("First five rows:")
display(df.head())
print("Descriptive statistics:")
display(df.describe().T)

## Missing values and duplicate rows

In [ ]:
missing_summary = df.isna().sum().rename("missing_values").to_frame()
print("Missing values by column:")
display(missing_summary)
duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows (beyond first occurrence): {duplicate_count}")

## Target distribution and binary label balance

In [ ]:
quality_counts = df["quality"].value_counts().sort_index()
label_counts = df["good"].value_counts().sort_index()
label_names = {0: "Not good (0)", 1: "Good (1)"}
print("Quality counts:")
display(quality_counts.rename("count").to_frame())
print("Binary label counts:")
display(label_counts.rename(index=label_names).rename("count").to_frame())

fig, ax = plt.subplots(figsize=(7, 4))
quality_counts.plot(kind="bar", ax=ax, color="#3b7a57")
ax.set(title="Wine quality score distribution", xlabel="Quality score", ylabel="Rows")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "quality_distribution.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
label_plot_counts = label_counts.rename(index=label_names)
label_plot_counts.plot(kind="bar", ax=ax, color=["#c77c3f", "#3b7a57"])
ax.set(title="Binary target balance", xlabel="Label", ylabel="Rows")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "binary_label_balance.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

## Feature distributions and outliers

The plots exclude both `quality` (the source of the label) and the derived `good` label. Each boxplot uses its feature's own scale so outliers remain visible across different units.

In [ ]:
feature_columns = [column for column in df.columns if column not in {"quality", "good"}]
feature_data = df[feature_columns]

axes = feature_data.hist(figsize=(14, 10), bins=25, color="#477c8a", edgecolor="white")
hist_figure = axes[0, 0].figure
hist_figure.suptitle("Feature distributions", y=1.01)
hist_figure.tight_layout()
hist_figure.savefig(FIGURE_DIR / "feature_histograms.png", dpi=150, bbox_inches="tight")
display(hist_figure)
plt.close(hist_figure)

rows = (len(feature_columns) + 2) // 3
box_figure, box_axes = plt.subplots(rows, 3, figsize=(14, 3 * rows))
for axis, column in zip(box_axes.flat, feature_columns):
    axis.boxplot(feature_data[column].dropna(), vert=False, showfliers=True)
    axis.set_title(column)
    axis.set_xlabel("Value")
for axis in box_axes.flat[len(feature_columns):]:
    axis.set_visible(False)
box_figure.suptitle("Feature boxplots (outliers shown)", y=1.01)
box_figure.tight_layout()
box_figure.savefig(FIGURE_DIR / "feature_boxplots.png", dpi=150, bbox_inches="tight")
display(box_figure)
plt.close(box_figure)

## Correlation heatmap and strongest label relationships

In [ ]:
correlation_data = df.drop(columns="quality")
correlation_matrix = correlation_data.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(correlation_matrix, cmap="vlag", center=0, annot=True, fmt=".2f", square=True, ax=ax)
ax.set_title("Feature and binary-label correlations (quality excluded)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

label_correlations = (
    correlation_matrix["good"]
    .drop(labels="good")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
print("Features most correlated with the binary label (Pearson correlation):")
display(label_correlations.head(10).rename("correlation_with_good").to_frame())
print(f"PNG figures saved in: {FIGURE_DIR}")

## Findings and decisions

- The source contains 1,599 rows and 12 columns; the derived binary label has 744 not-good rows and 855 good rows. This is a reasonably balanced baseline target, so no resampling is justified initially.
- There are 240 exact duplicate rows. Check their provenance and compare deduplicated versus retained data; do not allow duplicate copies to fall across train/test partitions, since that can inflate evaluation scores.
- `chlorides` and `residual sugar` are strongly right-skewed (skewness about 5.68 and 4.54). Inspect their boxplots; consider a transformation only if it improves validation results and is fitted within the training pipeline.
- Feature scales differ. Standardize features for logistic regression using a scaler fitted only on each training fold; random forests do not require scaling.
- Do not use `quality` as a predictor because it directly defines `good`. Keep all learned preprocessing inside the training split or cross-validation pipeline to prevent leakage.
- The strongest observed linear relationships with `good` are `alcohol` (about +0.435) and `volatile acidity` (about -0.321); correlation is descriptive, not evidence of causation.